In [24]:
print("hello")

hello


In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\data science\\Deep_learning_project\\kidney_disease_classification'

In [25]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [10]:
%pip install -e "e:/data science/Deep_learning_project/kidney_disease_classification"

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///E:/data%20science/Deep_learning_project/kidney_disease_classification
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for cnnClassifier (pyproject.toml): started
  Building editable for cnnClassifier (pyproject.toml): finished with status 'done'
  Created wheel for cnnClassifier: filename=cnnclassifier-0.0.0-0.editable-py3-none-any.whl size=2508 sha256=95cf24652e09d640bd193460d48b9477ab4f92606f2bb6cca44c8c07a8a58464
  Stored in dire


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [26]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [27]:
%pip install gdown


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [29]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''
        
        
        zip_download_dir = self.config.local_data_file
        
        # Check if file already exists
        if os.path.exists(zip_download_dir):
            logger.info(f"File already exists at {zip_download_dir}, skipping download")
            return zip_download_dir

        try: 
            dataset_url = self.config.source_URL
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id, zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
            return zip_download_dir

        except Exception as e:
            logger.error(f"Download failed: {str(e)}")
            logger.error(f"You can manually download from: {dataset_url}")
            logger.error("Please ensure the Google Drive link has sharing permissions set to 'Anyone with the link'")
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [30]:
import os

os.chdir("../")
print(os.getcwd())

e:\data science\Deep_learning_project


In [ ]:
try:

    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    
    print("✓ Data ingestion completed successfully!")
    
except Exception as e:
    print(f"✗ Error: {str(e)}")
    import traceback
    traceback.print_exc()
    raise e

[2026-06-17 10:21:01,032: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-17 10:21:01,034: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-17 10:21:01,035: INFO: common: created directory at: artifacts]
[2026-06-17 10:21:01,037: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-17 10:21:01,039: INFO: 3812171282: Downloading data from https://drive.google.com/file/d/1Wh3cfCrcKPzGAqhxdvS2xbGeAnjEtQN-/view?usp=drive_link into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1Wh3cfCrcKPzGAqhxdvS2xbGeAnjEtQN-
From (redirected): https://drive.google.com/uc?id=1Wh3cfCrcKPzGAqhxdvS2xbGeAnjEtQN-&confirm=t&uuid=c1cb73fc-0d74-46dc-ae7f-346e10b9c906
To: e:\data science\Deep_learning_project\kidney_disease_classification\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:10<00:00, 5.65MB/s]

[2026-06-17 10:21:14,044: INFO: 3812171282: Downloaded data from https://drive.google.com/file/d/1Wh3cfCrcKPzGAqhxdvS2xbGeAnjEtQN-/view?usp=drive_link into file artifacts/data_ingestion/data.zip]


✓ Data ingestion completed successfully!
